In [18]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_excel(r"C:\Users\ravik\Desktop\CSTR digital twin\data\combined_data_cleaned.xlsx")

In [36]:
df = df.iloc[:,1:]

In [37]:
df.sample(3)

,R_temp,R_vol,H2_flow,feed1_flow,frac_gly_in,PDO_fraction,gly_conv
1605,277.909677,875.973570,17.960445,199.335131,0.752875,0.082089,11.885837
3042,185.741778,1816.904686,13.535287,187.602263,0.157894,0.064280,43.648041
5388,184.921417,1900.165163,17.139344,171.828204,0.641728,0.088625,15.187881


In [41]:
x = df.iloc[:,0:5]
y = df.iloc[:,-1]

0       19.930615
1       28.271647
2       15.436754
3       31.320362
4       18.554813
          ...    
9795    20.600826
9796    64.495478
9797    19.346204
9798    18.448356
9799    24.871051
Name: gly_conv, Length: 9800, dtype: float64

In [46]:
from sklearn.model_selection import train_test_split

In [48]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size= 0.3)

In [50]:
from sklearn.ensemble import RandomForestRegressor

In [63]:
rfr= RandomForestRegressor()
rfr.fit(x_train,y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [64]:
y_pred = rfr.predict(x_test)

In [65]:
from sklearn.metrics import r2_score

In [66]:
r2_score(y_test,y_pred)

0.9973806307988972

In [67]:
cross_val_score(rfr,x,y, cv= 5,scoring='r2').mean()

np.float64(0.9992067429690493)

In [55]:
import xgboost as xgb

In [68]:
xgb_model = xgb.XGBRegressor(n_estimators = 200)

In [69]:
xgb_model.fit(x_train,y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [70]:
y_pred1 = xgb_model.predict(x_test)
r2_score(y_test,y_pred1)

0.9982947637880413

In [71]:
from sklearn.model_selection import cross_val_score

In [72]:
cross_val_score(xgb_model,x,y, cv= 5,scoring='r2').mean()

np.float64(0.9998410997757141)

## optuna for xgb

In [74]:
import optuna

In [78]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators',80,500)
    max_depth = trial.suggest_int("max_depth",5,50)

    model = xgb.XGBRegressor(
        n_estimators = n_estimators,
        max_depth = max_depth,
        random_state= 42
    )
    score =cross_val_score(model,x,y, cv= 5,scoring='r2').mean()
    
    return score
    

In [79]:
study =optuna.create_study(direction ='maximize')
study.optimize(objective,n_trials= 50)

print(study.best_params)
print(study.best_value)

[I 2026-06-12 20:21:02,370] A new study created in memory with name: no-name-87377704-b446-4aea-80f5-a04fddeba426
[I 2026-06-12 20:21:55,354] Trial 0 finished with value: 0.999999998261132 and parameters: {'n_estimators': 318, 'max_depth': 50}. Best is trial 0 with value: 0.999999998261132.
[I 2026-06-12 20:22:17,443] Trial 1 finished with value: 0.999999952662062 and parameters: {'n_estimators': 375, 'max_depth': 8}. Best is trial 0 with value: 0.999999998261132.
[I 2026-06-12 20:22:39,700] Trial 2 finished with value: 0.9999999952145577 and parameters: {'n_estimators': 340, 'max_depth': 16}. Best is trial 0 with value: 0.999999998261132.
[I 2026-06-12 20:23:07,041] Trial 3 finished with value: 0.9999999975670072 and parameters: {'n_estimators': 168, 'max_depth': 19}. Best is trial 0 with value: 0.999999998261132.
[I 2026-06-12 20:23:44,835] Trial 4 finished with value: 0.9999999981691794 and parameters: {'n_estimators': 233, 'max_depth': 23}. Best is trial 0 with value: 0.99999999826

{'n_estimators': 210, 'max_depth': 34}
0.9999999982799045


## Optuna for Randomforest

In [80]:
def objective(trial):

    n_estimators = trial.suggest_int('n_estimators',80,500)
    max_depth = trial.suggest_int("max_depth",5,50)

    model = RandomForestRegressor(
        n_estimators = n_estimators,
        max_depth = max_depth,
        random_state= 42
    )
    score =cross_val_score(model,x,y, cv= 5,scoring='r2').mean()
    
    return score
    
    

In [ ]:
study =optuna.create_study(direction ='maximize')
study.optimize(objective,n_trials= 30)

print(study.best_params)
print(study.best_value)

[I 2026-06-12 22:37:59,524] A new study created in memory with name: no-name-210b0375-bf5d-46f1-aa58-78bedbc42449
